# File structure thats expected:

##### Single experiment folder
##### - 01
##### ------Labels.tif
##### ------Cy3.5/Cy5.tif
##### ------DAPI.tif
##### - 02
##### - 03
##### - etc.

    

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from skimage import io
from skimage.morphology import medial_axis
from pathlib import Path

# --------------------------
# CONFIG
# --------------------------
FILENAME_PREFIX = "CET145"
DATASET_FOLDER = Path('/Volumes/A4_Eva/Eva/260526_Ca_Mono_SLR1KOandWHI3KO_ASH1_HWP1/smFISH/CET145/00_Analysed')
mRNA            = "HWP1"
CY_CHANNEL = 'CY3.5 NAR'  # change to 'CY5', 'CY3', etc.

BIN_SIZE        = 1    # µm per bin
PIXEL_SIZE      = 0.065  # µm per pixel

## Making masks

In [ ]:
import napari

viewer = napari.Viewer()

KEYWORD_COLORMAP = {
    'DAPI': 'blue',
    'CY5': 'green'
}

def set_layer_style(event):
    layer = event.value
    
    if hasattr(layer, 'rgb') and layer.rgb:
        layer.rgb = False
    
    layer.blending = 'additive'
    
    name_lower = layer.name.lower()
    for keyword, colormap in KEYWORD_COLORMAP.items():
        if keyword in name_lower:
            layer.colormap = colormap
            break

viewer.layers.events.inserted.connect(set_layer_style)

napari.run()

## Helper functions

In [ ]:
def geodesic_distance_map(mask, tip_coord, pixel_size_um=1.0):
    rows, cols = mask.shape
    G = nx.Graph()
    for y in range(rows):
        for x in range(cols):
            if mask[y, x]:
                G.add_node((y, x))
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx_ = y + dy, x + dx
                        if 0 <= ny < rows and 0 <= nx_ < cols:
                            if mask[ny, nx_]:
                                G.add_edge((y, x), (ny, nx_), weight=np.hypot(dy, dx))
    length_dict = nx.single_source_dijkstra_path_length(G, source=tip_coord, weight='weight')
    dist_map = np.zeros_like(mask, dtype=float)
    for (y, x), dist in length_dict.items():
        dist_map[y, x] = dist * pixel_size_um
    return dist_map

def build_skeleton_graph(skeleton):
    G = nx.Graph()
    rows, cols = skeleton.shape
    for y in range(rows):
        for x in range(cols):
            if skeleton[y, x]:
                G.add_node((y, x))
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx_ = y + dy, x + dx
                        if 0 <= ny < rows and 0 <= nx_ < cols:
                            if skeleton[ny, nx_]:
                                G.add_edge((y, x), (ny, nx_))
    return G

def extend_skeleton(skel, endpoint, max_steps=20):
    y, x = endpoint
    skel_extended = skel.copy()
    neighbors = [(y+dy, x+dx)
                 for dy in [-1,0,1] for dx in [-1,0,1]
                 if (dy!=0 or dx!=0) and 0<=y+dy<skel.shape[0] and 0<=x+dx<skel.shape[1] and skel[y+dy,x+dx]]
    if not neighbors:
        return skel_extended
    ny, nx = neighbors[0]
    dy, dx = y - ny, x - nx
    for _ in range(max_steps):
        y_new, x_new = int(round(y + dy)), int(round(x + dx))
        if not (0 <= y_new < skel.shape[0] and 0 <= x_new < skel.shape[1]):
            break
        if skel[y_new, x_new] == 0:
            skel_extended[y_new, x_new] = 1
            y, x = y_new, x_new
        else:
            break
    return skel_extended

##  Data Extraction Loop

In [ ]:
all_records  = []  # reset before each run
dapi_records = []  # reset before each run
subfolders   = sorted([f for f in DATASET_FOLDER.iterdir() if f.is_dir() and f.name.isdigit()])

# Sanity check
print(f"Dataset:    {DATASET_FOLDER}")
print(f"Subfolders: {[f.name for f in subfolders]}")

for subfolder in subfolders:
    imgs_cy = sorted(list(subfolder.glob(f'MAX_{FILENAME_PREFIX}_{CY_CHANNEL}*.tif')))
    imgs_dapi = sorted(list(subfolder.glob(f'MAX_{FILENAME_PREFIX}_DAPI*.tif')))
    labels    = sorted(list(subfolder.glob('CET145_CY3.5_Labels_*.tif')))

    if not imgs_cy or not imgs_dapi or not labels:
        print(f"[SKIP] {subfolder.name} — missing {CY_CHANNEL}, DAPI or Labels")
        continue

    print(f"\nProcessing {subfolder.name}/")

    zproj = io.imread(str(imgs_cy[0]))
    dapi      = io.imread(str(imgs_dapi[0]))
    label_img = io.imread(str(labels[0]))

    # Skeletonize
    skeleton_all = np.zeros_like(label_img, dtype=bool)
    for lab in np.unique(label_img)[1:]:
        mask = label_img == lab
        skeleton_all |= medial_axis(mask)

    # Extend skeleton
    G               = build_skeleton_graph(skeleton_all.astype(int))
    pixel_neighbors = {node: len(list(G.neighbors(node))) for node in G.nodes}
    endpoints       = [node for node, n in pixel_neighbors.items() if n == 1]

    extended_skeleton = skeleton_all.copy().astype(int)
    for ep in endpoints:
        extended_skeleton = extend_skeleton(extended_skeleton, ep, max_steps=20)
    extended_skeleton = np.where(label_img > 0, extended_skeleton, 0)

    # Rebuild graph on extended skeleton
    G2               = build_skeleton_graph(extended_skeleton)
    pixel_neighbors2 = {node: len(list(G2.neighbors(node))) for node in G2.nodes}
    endpoints2       = [node for node, n in pixel_neighbors2.items() if n == 1]

    # Find tip per label using DAPI centroid — tip is furthest endpoint from nucleus
    max_per_label = {}
    for label_id in np.unique(label_img)[1:]:
        cell_mask = label_img == label_id

        cell_endpoints = [ep for ep in endpoints2 if cell_mask[ep[0], ep[1]]]
        if len(cell_endpoints) < 2:
            continue

        # Find DAPI centroid within this cell mask
        dapi_within = dapi * cell_mask
        dapi_coords = np.argwhere(dapi_within > np.percentile(dapi_within[cell_mask], 75))
        if len(dapi_coords) == 0:
            print(f"  [WARN] label {label_id} — no DAPI signal found, skipping")
            continue
        nucleus_centroid = dapi_coords.mean(axis=0)  # (y, x)

        # Tip = endpoint furthest from nucleus centroid
        distances_to_nucleus = {
            ep: np.hypot(ep[0] - nucleus_centroid[0], ep[1] - nucleus_centroid[1])
            for ep in cell_endpoints
        }
        tip_coord = max(distances_to_nucleus, key=distances_to_nucleus.get)
        max_per_label[label_id] = (tip_coord, distances_to_nucleus[tip_coord])

        print(f"  [TIP] label {label_id} — tip at {tip_coord}, dist from nucleus: {distances_to_nucleus[tip_coord]:.1f}px")

    # Geodesic distance + binning
    max_distance  = 0
    distance_maps = {}
    for label_id, (tip_coord, _) in max_per_label.items():
        cell_mask = label_img == label_id
        if not np.any(cell_mask):
            continue
        dist_map = geodesic_distance_map(cell_mask, tip_coord, pixel_size_um=PIXEL_SIZE)
        distance_maps[label_id] = dist_map
        max_distance = max(max_distance, dist_map.max())

    bins        = np.arange(0, max_distance + BIN_SIZE, BIN_SIZE)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    for label_id, dist_map in distance_maps.items():
        cell_mask        = label_img == label_id
        coords           = np.argwhere(cell_mask)
        distances        = dist_map[coords[:,0], coords[:,1]]
        intensities      = zproj[coords[:,0], coords[:,1]]
        dapi_intensities = dapi[coords[:,0], coords[:,1]]
        bin_indices      = np.digitize(distances, bins) - 1

        for i, center in enumerate(bin_centers):
            mask = bin_indices == i
            if np.any(mask):
                all_records.append({
                    'label':     f"{subfolder.name}_{label_id}",
                    'distance':  center,
                    'intensity': np.mean(intensities[mask])
                })
                dapi_records.append({
                    'label':     f"{subfolder.name}_{label_id}",
                    'distance':  center,
                    'intensity': np.mean(dapi_intensities[mask])
                })

# Create dfs fresh at end of block
df      = pd.DataFrame(all_records)
df_dapi = pd.DataFrame(dapi_records)
print(f"\nTotal hyphae: {df['label'].nunique()} across {len(subfolders)} fields of view")

In [ ]:
print(df['label'].unique())
print(len(df))

## CY Channel Intensity Profiles

## Manual Direction Corrections (Only run when needed)

In [ ]:
# Add any hyphae that need flipping here as "subfolder_labelid"
FLIP_LABELS = ['06_1']  # example, change to the correct label. This one for monochannel G42 HWP1

for flip_label in FLIP_LABELS:
    # Flip in df_norm
    mask = df_norm['label'] == flip_label
    if not mask.any():
        print(f"[WARN] {flip_label} not found in df_norm")
    else:
        max_dist = df_norm.loc[mask, 'distance'].max()
        df_norm.loc[mask, 'distance'] = max_dist - df_norm.loc[mask, 'distance']
        print(f"[FLIP] {flip_label} flipped in df_norm, max distance was {max_dist:.2f} µm")

    # Flip in df
    mask = df['label'] == flip_label
    if not mask.any():
        print(f"[WARN] {flip_label} not found in df")
    else:
        max_dist = df.loc[mask, 'distance'].max()
        df.loc[mask, 'distance'] = max_dist - df.loc[mask, 'distance']
        print(f"[FLIP] {flip_label} flipped in df, max distance was {max_dist:.2f} µm")

    # Flip in df_dapi
    mask = df_dapi['label'] == flip_label
    if not mask.any():
        print(f"[WARN] {flip_label} not found in df_dapi")
    else:
        max_dist = df_dapi.loc[mask, 'distance'].max()
        df_dapi.loc[mask, 'distance'] = max_dist - df_dapi.loc[mask, 'distance']
        print(f"[FLIP] {flip_label} flipped in df_dapi, max distance was {max_dist:.2f} µm")

In [ ]:
print(f"Plotting {df['label'].nunique()} hyphae across {df['label'].str.split('_').str[0].nunique()} fields of view")
print(f"Distance range: 0 — {df['distance'].max():.1f} µm")
print(f"Intensity range: {df['intensity'].min():.0f} — {df['intensity'].max():.0f}")

plt.figure(figsize=(8, 5))
sns.lineplot(data=df, x='distance', y='intensity', hue='label',
             estimator=None, alpha=0.5, legend=False)
plt.xlabel('Distance from tip (µm)')
plt.ylabel('Average intensity per bin')
plt.title(f'{mRNA} intensity profiles — {FILENAME_PREFIX} ({df["label"].nunique()} hyphae)')
plt.grid(True)
plt.xlim(0, 40) # X-axis
plt.ylim(0, 17500) # Y-axis
plt.yticks(np.arange(0, 17501, 2500))  # 0, 2500, 5000, ..., 17500
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_intensity_profiles.pdf", bbox_inches='tight')
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_intensity_profiles.png", dpi=300, bbox_inches='tight')
plt.show()

# Normalize data based on highest intensity per hyphae

In [ ]:
collect = []

for ix, group in df.groupby('label'):
    group['intensity'] = group['intensity'] / group['intensity'].max()
    collect.append(group)
    # plt.plot(group['distance'], a)

df_norm = pd.concat(collect)

In [ ]:
print(f"Plotting {df_norm['label'].nunique()} hyphae across {df_norm['label'].str.split('_').str[0].nunique()} fields of view")
print(f"Distance range: 0 — {df_norm['distance'].max():.1f} µm")
print(f"Intensity range: {df_norm['intensity'].min():.0f} — {df_norm['intensity'].max():.0f}")

plt.figure(figsize=(8, 5))
sns.lineplot(data=df_norm, x='distance', y='intensity', hue='label',
             estimator=None, alpha=0.5, legend=False)
plt.xlabel('Distance from tip (µm)')
plt.ylabel('Average intensity per bin')
plt.title(f'{mRNA} intensity profiles — {FILENAME_PREFIX} ({df["label"].nunique()} hyphae) normalized')
plt.grid(True)
plt.xlim(0, 40) # X-axis
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_intensity_profiles_normalized.pdf", bbox_inches='tight')
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_intensity_profiles_normalized.png", dpi=300, bbox_inches='tight')
plt.show()

## DAPI Sanity Check

In [ ]:
bin_size     = BIN_SIZE
max_distance = df_dapi.groupby('label')['distance'].max().max()
bins         = np.arange(0, max_distance + bin_size, bin_size)
bin_centers  = (bins[:-1] + bins[1:]) / 2

hypha_ids  = sorted(df_dapi['label'].unique())
num_bins   = len(bin_centers)
num_hyphae = len(hypha_ids)

intensity_matrix_dapi = np.full((num_hyphae, num_bins), np.nan)
for i, hypha_id in enumerate(hypha_ids):
    group = df_dapi[df_dapi['label'] == hypha_id]
    for _, row in group.iterrows():
        j = np.searchsorted(bin_centers, row['distance'])
        if 0 <= j < num_bins:
            intensity_matrix_dapi[i, j] = row['intensity']

# Sort by hypha length
lengths      = (~np.isnan(intensity_matrix_dapi)).sum(axis=1)
sorted_order = np.argsort(lengths)
intensity_matrix_dapi_sorted = intensity_matrix_dapi[sorted_order]
sorted_labels = [hypha_ids[i] for i in sorted_order]

plt.figure(figsize=(12, max(6, num_hyphae * 0.3)))
im = plt.imshow(intensity_matrix_dapi_sorted, aspect='auto',
                extent=[0, bin_centers[-1], 0, num_hyphae],   #######altered
                origin='lower', cmap='Blues')
plt.xlim(0, 40) # X-axis
plt.colorbar(im, label='DAPI Intensity')
plt.xlabel('Distance from tip (µm)')
plt.title(f'{FILENAME_PREFIX} — DAPI sanity check (nucleus should peak at base)')

tick_positions = np.arange(num_hyphae) + 0.5
plt.yticks(tick_positions, sorted_labels, fontsize=7)

plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_DAPI_sanity.pdf", bbox_inches='tight')
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_DAPI_sanity.png", dpi=300, bbox_inches='tight')
plt.show()

## DAPI METRICS

In [ ]:
from skimage.filters import threshold_otsu

DAPI_THRESHOLD_OVERRIDE = None  # set to a value e.g. 500 to override, None for automatic

# Auto threshold using Otsu on all DAPI intensities
if DAPI_THRESHOLD_OVERRIDE is not None:
    DAPI_THRESHOLD = DAPI_THRESHOLD_OVERRIDE
    print(f"DAPI threshold: {DAPI_THRESHOLD} (manual override)")
else:
    DAPI_THRESHOLD = threshold_otsu(df_dapi['intensity'].values)
    print(f"DAPI threshold: {DAPI_THRESHOLD:.0f} (auto Otsu)")

dapi_metrics = []
for hypha_id, group in df_dapi.groupby('label'):

    # DAPI spread — µm of hypha with DAPI signal above threshold
    dapi_above  = group[group['intensity'] > DAPI_THRESHOLD]
    dapi_spread = dapi_above['distance'].max() - dapi_above['distance'].min() if len(dapi_above) > 1 else np.nan

    # DAPI centroid — intensity-weighted mean distance from tip
    dapi_centroid = np.average(group['distance'], weights=group['intensity']) if group['intensity'].sum() > 0 else np.nan

    dapi_metrics.append({
        'label':            hypha_id,
        'dapi_spread_um':   dapi_spread,
        'dapi_centroid_um': dapi_centroid
    })

df_dapi_metrics = pd.DataFrame(dapi_metrics)
df_dapi_metrics['strain'] = FILENAME_PREFIX

print(df_dapi_metrics[['label', 'dapi_spread_um', 'dapi_centroid_um']].to_string())
print(f"\nMedian DAPI spread:   {df_dapi_metrics['dapi_spread_um'].median():.2f} µm")
print(f"Median DAPI centroid: {df_dapi_metrics['dapi_centroid_um'].median():.2f} µm from tip")

df_dapi_metrics.to_csv(DATASET_FOLDER / f"{FILENAME_PREFIX}_dapi_metrics.csv", index=False)
print(f"Saved to {DATASET_FOLDER}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for ax, col, ylabel, title in zip(
    axes,
    ['dapi_spread_um', 'dapi_centroid_um'],
    ['DAPI spread (µm)', 'DAPI centroid distance from tip (µm)'],
    ['DAPI spread', 'DAPI centroid position']
):
    data = df_dapi_metrics[col].dropna()
    x    = np.ones(len(data)) + np.random.uniform(-0.05, 0.05, len(data))
    ax.scatter(x, data, alpha=0.6, edgecolors='black', linewidths=0.5, zorder=3)
    ax.hlines(data.median(), 0.85, 1.15, colors='red', linewidths=2, label=f"median: {data.median():.2f}")
    ax.set_xticks([1])
    ax.set_xticklabels([FILENAME_PREFIX])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle(f'{FILENAME_PREFIX} — DAPI metrics (threshold: {DAPI_THRESHOLD:.0f})', y=1.02)
plt.tight_layout()
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_dapi_metrics.pdf", bbox_inches='tight')
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_dapi_metrics.png", dpi=300, bbox_inches='tight')
plt.show()

# Normalize data = Philipp code

## Manual Direction Corrections (Only run when needed)

In [ ]:
# Add any hyphae that need flipping here as "subfolder_labelid"

FLIP_LABELS = ['04_3','01_3',  '01_1','03_6', '06_7','04_4','01_2','02_1', '03_4','04_1','02_5', '03_7', '01_6', '06_5']  # example, change to the correct label. This one for monochannel G42 HWP1


for flip_label in FLIP_LABELS:
    mask        = df_norm['label'] == flip_label
    if not mask.any():
        print(f"[WARN] {flip_label} not found in df")
        continue
    max_dist    = df_norm.loc[mask, 'distance'].max()
    df_norm.loc[mask, 'distance'] = max_dist - df_norm.loc[mask, 'distance']
    print(f"[FLIP] {flip_label} flipped, max distance was {max_dist:.2f} µm")

# Also flip in dapi df for consistency
for flip_label in FLIP_LABELS:
    mask = df_dapi['label'] == flip_label
    if not mask.any():
        continue
    max_dist = df_dapi.loc[mask, 'distance'].max()
    df_dapi.loc[mask, 'distance'] = max_dist - df_dapi.loc[mask, 'distance']

## Intensity Heatmap


In [ ]:
print(f"Max intensity in matrix: {np.nanmax(intensity_matrix_sorted):.0f}")
print(f"95th percentile: {np.nanpercentile(intensity_matrix_sorted, 95):.0f}")

In [ ]:
FILENAME_PREFIX = FILENAME_PREFIX

In [ ]:
import seaborn as sns
sns.set_context("talk")
bin_size     = BIN_SIZE
max_distance = df_norm.groupby('label')['distance'].max().max()
bins         = np.arange(0, max_distance + bin_size, bin_size)
bin_centers  = (bins[:-1] + bins[1:]) / 2
hypha_ids  = sorted(df_norm['label'].unique())
num_bins   = len(bin_centers)
num_hyphae = len(hypha_ids)
intensity_matrix = np.full((num_hyphae, num_bins), np.nan)
for i, hypha_id in enumerate(hypha_ids):
    group = df_norm[df_norm['label'] == hypha_id]
    for _, row in group.iterrows():
        j = np.searchsorted(bin_centers, row['distance'])
        if 0 <= j < num_bins:
            intensity_matrix[i, j] = row['intensity']
            
            
# Sort by hypha length
lengths      = (~np.isnan(intensity_matrix)).sum(axis=1)
sorted_order = np.argsort(lengths)
intensity_matrix_sorted = intensity_matrix[sorted_order]
# Create subfolder
heatmap_folder = DATASET_FOLDER / "heatmaps"
heatmap_folder.mkdir(exist_ok=True)
# Print intensity stats to help choose vmax
print(f"Max intensity in matrix:   {np.nanmax(intensity_matrix_sorted):.0f}")
print(f"95th percentile:           {np.nanpercentile(intensity_matrix_sorted, 95):.0f}")


# Save single plot vmax=1
VMAX = 1
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(intensity_matrix_sorted, aspect='auto',
               extent=[0, bin_centers[-1], 0, num_hyphae], #######altered
               origin='lower', cmap='plasma',
               vmin=0, vmax=VMAX,
               interpolation='None')
plt.xlim(0,40) # X-axis
plt.colorbar(im, ax=ax, label=f'{mRNA} Signal Intensity')
ax.set_xlabel('Distance from tip (µm)')
ax.set_ylabel('Hyphae (sorted by length)')
ax.set_title(f'{FILENAME_PREFIX} {mRNA} intensity heatmap')
plt.savefig(heatmap_folder / f"{FILENAME_PREFIX}_heatmap_vmax{VMAX}.pdf", bbox_inches='tight')
plt.savefig(heatmap_folder / f"{FILENAME_PREFIX}_heatmap_vmax{VMAX}.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved heatmap to {heatmap_folder}")

## Plot sanity check

In [ ]:
# Prints IDs to check!

bin_size     = 2
max_distance = df_norm.groupby('label')['distance'].max().max()
bins         = np.arange(0, max_distance + bin_size, bin_size)
bin_centers  = (bins[:-1] + bins[1:]) / 2
hypha_ids  = sorted(df_norm['label'].unique())
num_bins   = len(bin_centers)
num_hyphae = len(hypha_ids)

intensity_matrix = np.full((num_hyphae, num_bins), np.nan)
print(intensity_matrix.shape)
for i, hypha_id in enumerate(hypha_ids):
    group = df_norm[df_norm['label'] == hypha_id]
    for _, row in group.iterrows():
        j = np.searchsorted(bin_centers, row['distance'])
        if 0 <= j < num_bins:
            # print(j)
            intensity_matrix[i, j] = row['intensity']

# Sort by hypha length
lengths      = (~np.isnan(intensity_matrix)).sum(axis=1)
sorted_order = np.argsort(lengths)
intensity_matrix_sorted = intensity_matrix[sorted_order]
sorted_labels = [hypha_ids[i] for i in sorted_order]  # labels in sorted order

VMAX = 1
plt.figure(figsize=(12, max(6, num_hyphae * 0.3)))  # scale height to number of hyphae
im = plt.imshow(intensity_matrix_sorted, aspect='auto',
                extent=[0, bin_centers[-1], 0, num_hyphae], #####altered
                origin='lower', cmap='magma',
                vmin=0, vmax=VMAX)
plt.xlim(0, 40) # X-axis
plt.colorbar(im, label='Intensity')
plt.xlabel('Distance from tip (µm)')
plt.title(f'{FILENAME_PREFIX} — {mRNA} hyphal intensity heatmap')

# Add hypha ID labels on y-axis
tick_positions = np.arange(num_hyphae) + 0.5
plt.yticks(tick_positions, sorted_labels, fontsize=7)

plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_heatmap.pdf", bbox_inches='tight')
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_heatmap.png", dpi=300, bbox_inches='tight')
df_norm.to_csv(DATASET_FOLDER / f"{FILENAME_PREFIX}_intensity_profiles.csv", index=False)
print(f"Saved plot and CSV to {DATASET_FOLDER}")
plt.show()

## Tip to body ratio

- Ratio = 1 — equal intensity in both zones, no localization
- Ratio > 1 — tip brighter than body = localized
- Ratio < 1 — body brighter than tip

In [ ]:
TIP_ZONE_START  = 0    # µm
TIP_ZONE_END    = 5    # µm
BODY_ZONE_START = 5   # µm
BODY_ZONE_END   = 10   # µm — same 5 µm window as tip

ratios = []
for hypha_id, group in df.groupby('label'):
    tip_intensity  = group[(group['distance'] >= TIP_ZONE_START)  & (group['distance'] <= TIP_ZONE_END)]['intensity'].mean()
    body_intensity = group[(group['distance'] >= BODY_ZONE_START) & (group['distance'] <= BODY_ZONE_END)]['intensity'].mean()

    if pd.notna(tip_intensity) and pd.notna(body_intensity) and body_intensity > 0:
        ratios.append({
            'label':      hypha_id,
            'tip_mean':   tip_intensity,
            'body_mean':  body_intensity,
            'ratio':      tip_intensity / body_intensity
        })

df_ratio = pd.DataFrame(ratios)
df_ratio['strain'] = FILENAME_PREFIX

print(f"Hyphae included: {len(df_ratio)} / {df['label'].nunique()} total")
print(f"  (excluded hyphae shorter than {BODY_ZONE_END} µm — no body zone)")
print(df_ratio[['label', 'ratio']].to_string())
print(f"\nMedian tip:body ratio: {df_ratio['ratio'].median():.2f}")

df_ratio.to_csv(DATASET_FOLDER / f"{FILENAME_PREFIX}_tip_body_ratio.csv", index=False)
print(f"Saved to {DATASET_FOLDER}")

# Plot
plt.figure(figsize=(6, 4))
x = np.ones(len(df_ratio)) + np.random.uniform(-0.05, 0.05, len(df_ratio))
plt.scatter(x, df_ratio['ratio'], alpha=0.6, edgecolors='black', linewidths=0.5, zorder=3)
plt.axhline(df_ratio['ratio'].median(), color='red', linestyle='-', linewidth=1.5,
            label=f"median: {df_ratio['ratio'].median():.2f}")
plt.axhline(1, color='gray', linestyle='--', linewidth=0.8)
plt.xticks([1], [FILENAME_PREFIX])
plt.ylabel('Tip:body intensity ratio')
plt.title(f'{FILENAME_PREFIX} — tip:body ratio\n'
          f'(tip {TIP_ZONE_START}-{TIP_ZONE_END}µm vs body {BODY_ZONE_START}-{BODY_ZONE_END}µm)')
plt.legend()
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_tip_body_ratio.pdf", bbox_inches='tight')
plt.savefig(DATASET_FOLDER / f"{FILENAME_PREFIX}_tip_body_ratio.png", dpi=300, bbox_inches='tight')
plt.show()